# Weeks 4–5 · Exercises

Each exercise is a function; run the check cell beneath it. The shared loader cell must run first.
Solutions: `solutions/02_solutions.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
DATA = next(p / "data" / "generated" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "make_data.py").exists())
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
print("pandas", pd.__version__)

In [ ]:
def load_clean_students():
    """Shared loader used by several exercises."""
    raw = pd.read_csv(DATA / "students.csv")
    return (raw.assign(major=lambda d: d["major"].str.strip(),
                       city=lambda d: d["city"].str.title(),
                       gpa=lambda d: pd.to_numeric(d["gpa"], errors="coerce"))
               .drop_duplicates().reset_index(drop=True))
students = load_clean_students()
enr = pd.read_csv(DATA / "enrollments.csv")
sales = pd.read_csv(DATA / "sales_2025.csv", parse_dates=["date"])
print(students.shape, enr.shape, sales.shape)

### Exercise 1 · Selection

Return a DataFrame with columns `name, gpa` for third- and fourth-year students in Irbid with GPA ≥ 3.5, sorted by GPA descending then name ascending.

In [ ]:
def top_irbid(df):
    ...

In [ ]:
r = top_irbid(students)
assert list(r.columns) == ["name", "gpa"]
assert (r["gpa"] >= 3.5).all() and r["gpa"].is_monotonic_decreasing
assert set(students.loc[r.index, "city"]) == {"Irbid"} and set(students.loc[r.index, "year"]) <= {3, 4}
print("✓ exercise 1 -", len(r), "rows")

### Exercise 2 · Cleaning audit

`audit(raw)` takes the **raw** `students.csv` frame and returns a dict with keys `duplicates`, `bad_gpa` (count of non-numeric, non-empty GPA tokens such as `unknown`), `blank_gpa` (count of NaN/empty), `untrimmed_major` (rows whose major has leading/trailing spaces), `upper_city` (rows whose city is all upper-case).

In [ ]:
def audit(raw):
    ...

In [ ]:
raw = pd.read_csv(DATA / "students.csv")
a = audit(raw)
assert set(a) == {"duplicates", "bad_gpa", "blank_gpa", "untrimmed_major", "upper_city"}
assert a["duplicates"] == 5 and a["bad_gpa"] > 0 and a["blank_gpa"] > 0 and a["untrimmed_major"] > 0 and a["upper_city"] > 0
print("✓ exercise 2", a)

### Exercise 3 · Missing values, group-wise

`fill_gpa_by_major(df)` returns a copy where missing GPAs are replaced by the **median GPA of the same major**. No other column changes.

In [ ]:
def fill_gpa_by_major(df):
    ...

In [ ]:
f = fill_gpa_by_major(students)
assert f["gpa"].isna().sum() == 0 and students["gpa"].isna().sum() > 0
m = students["gpa"].isna()
for major in students.loc[m, "major"].unique():
    med = students.loc[students["major"] == major, "gpa"].median()
    assert (f.loc[m & (students["major"] == major), "gpa"] == med).all()
print("✓ exercise 3")

### Exercise 4 · Derived columns

Add two columns: `progress` = credits / 132 as a percentage rounded to 1 decimal, and `band` = `pd.cut` of GPA into `[0,2)→'C'`, `[2,3)→'B'`, `[3,4]→'A'` (use `right=False` and include 4.0 in 'A'). Return the new frame.

In [ ]:
def derive(df):
    ...

In [ ]:
d = derive(students)
assert "progress" in d and "band" in d
assert abs(d.loc[0, "progress"] - round(students.loc[0, "credits"] / 132 * 100, 1)) < 1e-9
assert d.loc[d["gpa"] == 4.0, "band"].astype(str).eq("A").all()
assert d.loc[d["gpa"] < 2.0, "band"].astype(str).eq("C").all()
print("✓ exercise 4", d["band"].value_counts().to_dict())

### Exercise 5 · GroupBy summary

`city_report(df)` returns a DataFrame indexed by city with columns `students` (count), `mean_gpa` (2 dp), `top_major` (the most common major in that city), sorted by `students` descending.

In [ ]:
def city_report(df):
    ...

In [ ]:
r = city_report(students)
assert list(r.columns) == ["students", "mean_gpa", "top_major"]
assert r["students"].is_monotonic_decreasing and r["students"].sum() == len(students)
assert r.loc["Amman", "top_major"] == students[students["city"] == "Amman"]["major"].mode()[0]
print("✓ exercise 5"); display(r)

### Exercise 6 · Merge

`grades_with_names(students, enr)` returns one row per enrollment with columns `student_id, name, major, course_code, grade`, keeping **only** enrollments whose student exists in `students`, and dropping rows with a missing grade.

In [ ]:
def grades_with_names(students, enr):
    ...

In [ ]:
g = grades_with_names(students, enr)
assert list(g.columns) == ["student_id", "name", "major", "course_code", "grade"]
assert g["grade"].notna().all() and g["name"].notna().all()
assert len(g) == enr["grade"].notna().sum()
print("✓ exercise 6 -", len(g), "rows")

### Exercise 7 · Pivot

Using the result of Exercise 6, return a pivot table of **mean grade** with majors as rows and course codes as columns, rounded to 1 dp, plus a final column `overall` with each major's mean across all its enrollments (not the mean of the row means).

In [ ]:
def grade_pivot(g):
    ...

In [ ]:
p = grade_pivot(grades_with_names(students, enr))
assert "overall" in p.columns and p.index.name == "major"
ds = grades_with_names(students, enr).query("major == 'Data Science'")["grade"].mean().round(1)
assert abs(p.loc["Data Science", "overall"] - ds) < 1e-9
print("✓ exercise 7"); display(p)

### Exercise 8 · Time series

`monthly_product_revenue(sales)` returns a DataFrame indexed by month-end (`'ME'`) with one column per product, holding total revenue (units × unit_price) rounded to 0 dp.

In [ ]:
def monthly_product_revenue(sales):
    ...

In [ ]:
m = monthly_product_revenue(sales)
assert m.shape == (12, 5) and set(m.columns) == set(sales["product"].unique())
assert m.index[0].month == 1 and m.index[-1].month == 12
tot = (sales["units"] * sales["unit_price"]).sum()
assert abs(m.values.sum() - tot) < 5
print("✓ exercise 8"); display(m.head(3))

### Exercise 9 · Rolling + ranking

`best_week(sales)` returns the `Timestamp` of the **end** of the 7-day window with the highest total revenue (daily totals → rolling sum of 7 → argmax).

In [ ]:
def best_week(sales):
    ...

In [ ]:
b = best_week(sales)
assert isinstance(b, pd.Timestamp)
daily = sales.assign(rev=sales["units"] * sales["unit_price"]).groupby("date")["rev"].sum()
assert b == daily.rolling(7).sum().idxmax()
print("✓ exercise 9", b.date())

### Exercise 10 · Mini EDA function

`eda(df)` returns a dict with: `rows`, `cols`, `missing` (dict column→count, only columns with missing values), `numeric` (list of numeric column names), `top_categories` (dict: for each object/category column with ≤ 10 unique values, the most frequent value).

In [ ]:
def eda(df):
    ...

In [ ]:
e = eda(students)
assert e["rows"] == len(students) and e["cols"] == students.shape[1]
assert e["missing"] == {"gpa": int(students["gpa"].isna().sum())}
assert set(e["numeric"]) == {"student_id", "year", "gpa", "credits"}
assert set(e["top_categories"]) == {"major", "city"}
print("✓ exercise 10", e)